In [ ]:
import json
import os
import sys
import random
import pandas as pd
import numpy as np
import ast

# Env vars from .env at repo root (see .env.example).
HOME_DIR = os.environ.get("PROJECT_ROOT", os.getcwd())
DATA_DIR = os.environ.get("DATA_DIR", os.path.join(HOME_DIR, "data"))
INTERMEDIATE_DIR = os.environ.get(
    "LOGIC_Q_INTERMEDIATE_DIR",
    os.path.join(HOME_DIR, "logs", "logic_q_intermediates"),
)


# Data preprocessing

In [ ]:
SL_PARENT = os.path.join(HOME_DIR, "logic_q_mt", "data_generation")
if SL_PARENT not in sys.path:
    sys.path.append(SL_PARENT)
import importlib
from SimpleLogic import verify_results
importlib.reload(verify_results)
from SimpleLogic.verify_results import verify_row


In [ ]:
import glob

# Discover stage-1/2 shard dirs instead of hardcoding the count. We used 30
# shards (new_0_500k .. new_29_500k) for the released data; this glob works for
# any number produced by prepare_shards.py + the stage-1/2 commands.
shard_dirs = sorted(glob.glob(os.path.join(INTERMEDIATE_DIR, "new_*_500k")))
assert shard_dirs, (
    f"No shard dirs (new_*_500k/) under {INTERMEDIATE_DIR}. "
    "Run stage 0 (prepare_shards.py) and stages 1-2 first."
)

new_df = {}
for idx, shard_dir in enumerate(shard_dirs):
    print(f"Processing folder: {os.path.basename(shard_dir)}")
    counter = {"verified": {1: 0}, "failed (goal inferred from context)": {1: 0}, "failed (insufficient branch)": {1: 0}, "failed (not minimal)": {1: 0}, "failed (not globally minimal)": {1: 0}}
    src = os.path.join(shard_dir, "simplelogic_heldout_k_sufficient_data_new.csv")
    data = pd.read_csv(src)
    for _, row in data.iterrows():
        if verify_row(row, counter=counter, verbose=False):
            new_df.setdefault(idx, []).append(row)
    for key, value in counter.items():
        print(f"	{key}: {value}")


In [ ]:
new_df = pd.concat(sum(new_df.values(), []), axis=1).T
new_df["all_valid_qs"] = new_df.apply(lambda row: set([var[4:] if var.startswith("not ") else var for var in sum(ast.literal_eval(row["rules"]), [])]) - set(ast.literal_eval(row["known_facts"]) + [row["goal"]]), axis=1)
new_df["all_valid_qs"] = new_df["all_valid_qs"].apply(sorted)

In [ ]:
os.makedirs(INTERMEDIATE_DIR, exist_ok=True)
new_df.to_pickle(os.path.join(INTERMEDIATE_DIR, "data_new.pkl"))


## 1. Filter for simple stats

In [ ]:
new_df = pd.read_pickle(os.path.join(INTERMEDIATE_DIR, "data_new.pkl"))
new_df["k"].value_counts()


In [ ]:
data = pd.concat([
    new_df.query("k == 1 and (max_depth >= 4 and min_num_rules_needed >= 4 and num_vars >= 20)").sample(n=20000, replace=False, random_state=42),
    new_df.query("k == 2 and (max_depth >= 5 and min_num_rules_needed >= 5 and num_vars >= 30)").sample(n=5000, replace=False, random_state=42),
    new_df.query("k >= 3"),
]).reset_index(drop=True)
assert (data["known_untrue_facts"].apply(len) > 2).sum() == 0
data["k"].value_counts()

In [ ]:
data["all_valid_qs"] = data["all_valid_qs"].apply(sorted)

In [ ]:
os.makedirs(DATA_DIR, exist_ok=True)
data.to_csv(os.path.join(DATA_DIR, "logic_q_mt_filtered.csv"), index=False)


## 2. Add all alternative gts

**Stage 3 (sub-step): annotate all alternative ground-truth question sets.**

`add_all_alternative_gts.py` reads `logic_q_mt_filtered.csv` (written by the cell
above) and writes `logic_q_mt_filtered_with_alt.csv` (read by the cell below).

You can run it **either** by executing the next code cell, **or** manually from
the repository root:

```bash
PYTHONPATH=logic_q_mt/data_generation python \
    logic_q_mt/data_generation/SimpleLogic/add_all_alternative_gts.py \
    --in_csv  "$DATA_DIR/logic_q_mt_filtered.csv" \
    --out_csv "$DATA_DIR/logic_q_mt_filtered_with_alt.csv"
```

If you ran it manually, skip the next cell.

In [ ]:
# Runs SimpleLogic/add_all_alternative_gts.py (the shell command in the cell
# above). Skip this cell if you already ran that command manually.
import subprocess

subprocess.run(
    [
        sys.executable,
        os.path.join(SL_PARENT, "SimpleLogic", "add_all_alternative_gts.py"),
        "--in_csv", os.path.join(DATA_DIR, "logic_q_mt_filtered.csv"),
        "--out_csv", os.path.join(DATA_DIR, "logic_q_mt_filtered_with_alt.csv"),
    ],
    env={**os.environ, "PYTHONPATH": SL_PARENT},
    check=True,
)


## 3. Add all inferrable variable values

In [ ]:
data = pd.read_csv(os.path.join(DATA_DIR, "logic_q_mt_filtered_with_alt.csv"))


In [ ]:
import importlib
if SL_PARENT not in sys.path:
    sys.path.append(SL_PARENT)
from SimpleLogic import horn_sat_utils
importlib.reload(horn_sat_utils)
from SimpleLogic.horn_sat_utils import solve_unit_prop, parse_clauses, get_inferrable_var_values
from itertools import product
from typing import List

def get_gt_qs_inferred_var_values(rules: List[List[str]], known_facts: List[str], gt_qs: List[str], goal: str):
    clauses = parse_clauses(rules)
    all_vars = sorted(set([var[4:] if var.startswith("not ") else var for var in sum(rules, [])]))
    gt_qs_to_vars = {}
    for gt_qs_assign in product([True, False], repeat=len(gt_qs)):
        full_facts = known_facts + [f"{gt_qs[i]}" if gt_qs_assign[i] else f"not {gt_qs[i]}" for i in range(len(gt_qs))]
        valid_vars_values = get_inferrable_var_values(clauses, full_facts, all_vars, keep_fact_vars=False, use_int=True)
        gt_qs_to_vars[tuple(gt_qs_assign)] = valid_vars_values
    return gt_qs_to_vars

data["inferred_variable_values"] = data.apply(lambda row: {tuple(sorted(gt_qs)): get_gt_qs_inferred_var_values(
    ast.literal_eval(row["rules"]),
    ast.literal_eval(row["known_facts"]),
    sorted(gt_qs),
    row["goal"]
) for gt_qs in ast.literal_eval(row["gt_qs"])}, axis=1)


In [9]:
data_sampled = data.query("num_all_valid_qs_forbid_alternatives >= 10 and max_depth >= 4 and min_num_rules_needed >= 4").groupby("k").sample(n=200, replace=False, random_state=42).reset_index(drop=True)

In [10]:
len(ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])), ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])

(8,
 [{'["adorable", "straightforward", "versatile"]': {'target_value': 'tame',
    'derivation': {'leaf_words': {'adorable': 1,
      'versatile': 3,
      'straightforward': 1},
     'ancestor_words': {'tame': 0, 'hypocritical': 1, 'light': 2},
     'derivation': [[['straightforward', 'hypocritical', 'adorable'], 'tame'],
      [['light'], 'hypocritical'],
      [['versatile'], 'light']],
     'root_word': 'tame',
     'layer': 3}},
   '["adorable", "straightforward", "not versatile"]': {'target_value': 'not tame',
    'derivation': {'leaf_words': {'not versatile': 1},
     'ancestor_words': {'not tame': 0},
     'derivation': [[['not versatile'], 'not tame']],
     'root_word': 'not tame',
     'layer': 1}},
   '["adorable", "not straightforward", "versatile"]': {'target_value': 'not tame',
    'derivation': {'leaf_words': {'adorable': 2,
      'versatile': 6,
      'not straightforward': 2},
     'ancestor_words': {'not tame': 0,
      'not thoughtful': 1,
      'frail': 2,
      '

In [11]:
rng = random.Random(42)

def sample_consistent_indices(row):
    cols = ["gt_qs", "gt_q_to_derivations_min_depth", "gt_q_to_derivations_min_rules"]
    parsed_lists = [ast.literal_eval(row[c]) for c in cols]
    idx = rng.randrange(len(parsed_lists[0]))
    return pd.Series([json.dumps([lst[idx]]) for lst in parsed_lists], index=cols)

data_sampled[["gt_qs", "gt_q_to_derivations_min_depth", "gt_q_to_derivations_min_rules"]] = data_sampled.apply(sample_consistent_indices, axis=1)

In [15]:
len(ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])), len(ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])[0]), ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])

(1,
 8,
 [{'["light", "straightforward", "talkative"]': {'target_value': 'tame',
    'derivation': {'leaf_words': {'light': 5,
      'straightforward': 3,
      'talkative': 2},
     'ancestor_words': {'tame': 0,
      'adorable': 1,
      'calm': 2,
      'courageous': 3,
      'hypocritical': 4},
     'derivation': [[['straightforward', 'hypocritical', 'adorable'], 'tame'],
      [['calm', 'talkative', 'light'], 'adorable'],
      [['light'], 'hypocritical'],
      [['courageous', 'straightforward'], 'calm'],
      [['hypocritical'], 'courageous'],
      [['light'], 'hypocritical']],
     'root_word': 'tame',
     'layer': 5}},
   '["light", "straightforward", "not talkative"]': {'target_value': 'not tame',
    'derivation': {'leaf_words': {'light': 4, 'not talkative': 1},
     'ancestor_words': {'not tame': 0,
      'thoughtless': 1,
      'worried': 2,
      'hypocritical': 3},
     'derivation': [[['thoughtless', 'not talkative'], 'not tame'],
      [['worried'], 'thoughtless'],
 

In [25]:
rng = random.Random(42)

def sample_assignments_per_world(row):
    """Sample one assignment per world (target_value) for both columns with same assignments"""
    cell_value_min_depth = row["gt_q_to_derivations_min_depth"]
    cell_value_min_rules = row["gt_q_to_derivations_min_rules"]
    
    # Parse the JSON string
    data_min_depth = json.loads(cell_value_min_depth)
    data_min_rules = json.loads(cell_value_min_rules)
        
    # all variable sets
    assignments_list_min_depth = []
    assignments_list_min_rules = []
    for assignment_dict_min_depth, assignment_dict_min_rules in zip(data_min_depth, data_min_rules):
        # Group assignments by target_value
        # for each variable set
        true_assignments_min_depth = []
        false_assignments_min_depth = []
        true_assignments_min_rules = []
        false_assignments_min_rules = []
        for (assignment_key_min_depth, assignment_data_min_depth), (assignment_key_min_rules, assignment_data_min_rules) in zip(assignment_dict_min_depth.items(), assignment_dict_min_rules.items()):
            assert assignment_key_min_depth == assignment_key_min_rules
            if not assignment_data_min_depth['target_value'].startswith('not '):  # <target>
                true_assignments_min_depth.append((assignment_key_min_depth, assignment_data_min_depth))
                true_assignments_min_rules.append((assignment_key_min_rules, assignment_data_min_rules))
            else:  # not <target>
                parsed_key = json.loads(assignment_key_min_depth)
                not_count = sum(1 for var_value in parsed_key if var_value.startswith('not '))
                if not_count > 1:
                    continue
                false_assignments_min_depth.append((assignment_key_min_depth, assignment_data_min_depth))
                false_assignments_min_rules.append((assignment_key_min_rules, assignment_data_min_rules))
    
        # Sample one assignment per world (target_value)
        chosen_true_idx = rng.randint(0, len(true_assignments_min_depth) - 1) if true_assignments_min_depth else None
        chosen_false_idx = rng.randint(0, len(false_assignments_min_depth) - 1) if false_assignments_min_depth else None

        # Sample using the same keys for both columns
        sampled_min_depth = []
        sampled_min_rules = []
        if chosen_true_idx is not None:
            sampled_min_depth.append(true_assignments_min_depth[chosen_true_idx])
            sampled_min_rules.append(true_assignments_min_rules[chosen_true_idx])
        if chosen_false_idx is not None:
            sampled_min_depth.append(false_assignments_min_depth[chosen_false_idx])
            sampled_min_rules.append(false_assignments_min_rules[chosen_false_idx])
    
        assignments_list_min_depth.append(dict(sampled_min_depth))
        assignments_list_min_rules.append(dict(sampled_min_rules))
    
    results = [json.dumps(assignments_list_min_depth), json.dumps(assignments_list_min_rules)]
    
    return pd.Series(results, index=["gt_q_to_derivations_min_depth", "gt_q_to_derivations_min_rules"])

# Apply to both columns together to ensure same sampling
data_sampled[["gt_q_to_derivations_min_depth", "gt_q_to_derivations_min_rules"]] = data_sampled.apply(sample_assignments_per_world, axis=1)

In [26]:
len(ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])), len(ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])[0]), ast.literal_eval(data_sampled.iloc[400]["gt_q_to_derivations_min_rules"])

(1,
 2,
 [{'["light", "straightforward", "talkative"]': {'target_value': 'tame',
    'derivation': {'leaf_words': {'light': 5,
      'straightforward': 3,
      'talkative': 2},
     'ancestor_words': {'tame': 0,
      'adorable': 1,
      'calm': 2,
      'courageous': 3,
      'hypocritical': 4},
     'derivation': [[['straightforward', 'hypocritical', 'adorable'], 'tame'],
      [['calm', 'talkative', 'light'], 'adorable'],
      [['light'], 'hypocritical'],
      [['courageous', 'straightforward'], 'calm'],
      [['hypocritical'], 'courageous'],
      [['light'], 'hypocritical']],
     'root_word': 'tame',
     'layer': 5}},
   '["light", "straightforward", "not talkative"]': {'target_value': 'not tame',
    'derivation': {'leaf_words': {'light': 4, 'not talkative': 1},
     'ancestor_words': {'not tame': 0,
      'thoughtless': 1,
      'worried': 2,
      'hypocritical': 3},
     'derivation': [[['thoughtless', 'not talkative'], 'not tame'],
      [['worried'], 'thoughtless'],
 

In [ ]:
data_sampled["sample_id"] = data_sampled.index.values
data_sampled.to_csv(os.path.join(DATA_DIR, "logic_q_mt.csv"), index=False)
